# 02 — Baseline detector training (v1)

Giai đoạn 2 (`docs/PLAN.md`) — spec: `docs/specs/g2-baseline-training.md`.

Mục tiêu: train baseline YOLOv8 (`yolov8n.pt`) trên dataset v1 (5 lớp, 1013
ảnh — số liệu thật từ Giai đoạn 1), ghi lại mAP@0.5 / mAP@0.5:0.95 làm mốc
so sánh cho ablation (Giai đoạn 3) và retrain sau cải tiến (Giai đoạn 5).

**Chạy trên Colab** (cần GPU — Runtime → Change runtime type → T4 GPU).

## Setup — mount Drive + cd vào repo

Giống hệt cell setup ở `01_data_exploration.ipynb` — bắt buộc để
`weights/best.pt` train ra tự động nằm trong Drive (không mất khi hết
session), và để `data/raw/yoga_v1/` (đã tải ở Giai đoạn 1) tìm thấy được.

In [ ]:
import os

REPO_DIR_NAME = "computer-vision-project"  # đổi nếu bạn git clone ra tên thư mục khác

try:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_path = f"/content/drive/MyDrive/{REPO_DIR_NAME}"
    if not os.path.isdir(drive_path):
        raise FileNotFoundError(
            f"{drive_path} không tồn tại — kiểm tra lại bạn đã `git clone` repo vào "
            "đúng chỗ trong Drive chưa (T0.4), hoặc sửa REPO_DIR_NAME ở trên cho khớp."
        )
    os.chdir(drive_path)
except ImportError:
    pass  # không chạy trên Colab (vd Jupyter local) — giả định cwd đã là repo root

print("cwd:", os.getcwd())
assert os.path.isdir("scripts") and os.path.isdir("data"), (
    "Chưa đứng ở repo root — không thấy scripts/ và data/ ở cwd hiện tại."
)
assert os.path.isfile("data/raw/yoga_v1/data.yaml"), (
    "data/raw/yoga_v1/data.yaml chưa có — chạy notebooks/01_data_exploration.ipynb "
    "(Giai đoạn 1) trước để tải dataset."
)

## Đồng bộ code mới nhất

Thư mục Drive được clone 1 lần ở T0.4 — nếu code trên GitHub đã cập nhật
sau đó (vd `src/models/train.py` mới thêm), cell này `git pull` để lấy về.
An toàn chạy lại nhiều lần.

In [ ]:
import os

assert os.path.isdir(".git"), (
    "cwd hiện tại không phải repo root (không thấy .git) — runtime Colab có "
    "thể vừa bị reset. Chạy lại cell 'Setup — mount Drive + cd vào repo' ở "
    "trên (mount + cd) trước, rồi chạy lại cell này."
)
!git pull

## Cài dependencies

Mỗi phiên Colab mới đều mất hết package đã cài — cell này idempotent, chạy
lại nhiều lần không sao.

In [ ]:
!pip install -q -r requirements.txt
import torch
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print(
        "CẢNH BÁO: không thấy GPU — vào Runtime > Change runtime type > "
        "chọn T4 GPU, rồi chạy lại từ đầu notebook. Train trên CPU sẽ rất chậm."
    )

## T2.1–T2.2 — Train baseline

`train()` (`src/models/train.py`) là wrapper mỏng quanh
`ultralytics.YOLO(...).train(...)`, cố định seed + augmentation đã chốt ở
T1.5. Baseline: `yolov8n.pt`, `seed=42`, `epochs=50`, `imgsz=640` (lý do chốt
epochs=50 xem `docs/specs/g2-baseline-training.md`).

In [ ]:
from src.models.train import train

results = train(
    data="data/raw/yoga_v1/data.yaml",
    model="yolov8n.pt",
    seed=42,
    epochs=50,
    name="yolov8n_v1_baseline",
)
print("save_dir:", results.save_dir)

## T2.3 — Xác nhận output

In [ ]:
from pathlib import Path

save_dir = Path(results.save_dir)
expected = ["results.csv", "results.png", "weights/best.pt", "weights/last.pt"]
for rel in expected:
    p = save_dir / rel
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

## T2.4 — Weights nằm trong Drive

Vì cell Setup ở đầu notebook đã `cd` vào repo trong Drive, `results.save_dir`
phải tự động nằm dưới `/content/drive/...` — không cần copy tay.

In [ ]:
resolved = save_dir.resolve()
print("save_dir (resolved):", resolved)
if "drive" in str(resolved).lower():
    print("OK — nằm trong Google Drive, không mất khi hết session.")
else:
    print(
        "CẢNH BÁO: save_dir không có vẻ nằm trong Drive — kiểm tra lại cell "
        "Setup ở đầu notebook đã cd đúng chỗ chưa trước khi train."
    )

## T2.5 — mAP baseline

`train()` đã tự chạy 1 lượt validate trên `best.pt` ở cuối training —
`results` (cell T2.1–T2.2) chính là kết quả đó, không cần load lại
checkpoint và validate thêm lần nữa (tốn thời gian session Colab free-tier
vô ích).

In [ ]:
print("mAP@0.5:", results.box.map50)
print("mAP@0.5:0.95:", results.box.map)

**mAP baseline (điền sau khi chạy):**

- mAP@0.5: ___
- mAP@0.5:0.95: ___